# ManimFlow

Autonomous animation generator powered by [Aider](https://aider.chat) — **no API keys required**.

## Usage
1. Run cell 1 (Install) — takes ~3.5 min on first run, then restart runtime
2. After restart, run all remaining cells
3. Describe your animation and click **Generate**

In [ ]:
import importlib.util, sys, subprocess, platform, os

if platform.system() == 'Linux':
    if subprocess.run(['dpkg', '-s', 'libpango1.0-dev'], capture_output=True).returncode != 0:
        print("Installing system deps (~60s)...")
        subprocess.run(['apt-get', 'install', '-y', '-q',
            'libcairo2-dev', 'libpango1.0-dev', 'ffmpeg',
            'texlive-latex-base', 'texlive-latex-extra',
            'texlive-fonts-recommended', 'texlive-plain-generic', 'dvisvgm'], check=True)
elif platform.system() == 'Darwin':
    sdk = subprocess.run(['xcrun', '--show-sdk-path'], capture_output=True, text=True).stdout.strip()
    if sdk:
        os.environ.setdefault('SDKROOT', sdk)
    for pkg in ['cairo', 'pango', 'ffmpeg']:
        if subprocess.run(['brew', 'list', pkg], capture_output=True).returncode != 0:
            subprocess.run(['brew', 'install', pkg])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4', 'scipy==1.13.1'], check=True)

if not importlib.util.find_spec('manim'):
    print("Installing manim (~40s)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'manim', '-q'], check=True)

if not importlib.util.find_spec('aider'):
    print("Installing aider (~85s)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'aider-chat', '-q'], check=True)

print("Done. Restart the runtime (Runtime → Restart session), then run all cells again.")

In [ ]:
import pathlib, subprocess, re, json, threading, urllib.request
from http.server import HTTPServer, BaseHTTPRequestHandler
from socketserver import ThreadingMixIn

BASE = pathlib.Path.cwd()
MANIM_OUTPUT = BASE / 'manim_output'
MANIM_OUTPUT.mkdir(parents=True, exist_ok=True)

EXA_URL = "https://demos.exa.ai/chatbot-demo/api/chat/stream"
LOCAL_PORT = 18642

def _strip_followups(text):
    i = text.find("\n\n```followups")
    if i >= 0:
        text = text[:i]
    return re.sub(r'\n\[(["\']).*?\1(?:,\s*(["\']).*?\2)*\s*\]\s*$', '', text, flags=re.DOTALL).rstrip()

def _collect_exa_stream(messages):
    system_msg = next((m for m in messages if m['role'] == 'system'), None)
    non_system = [m for m in messages if m['role'] != 'system']
    last = non_system[-1]
    sys_content = system_msg['content'] if system_msg else ''
    user_content = "IMPORTANT: use plain ``` for ALL code blocks, never ```python or ```bash.\n\n" + sys_content + "\n\n" + last['content']
    payload = json.dumps({
        'message': user_content,
        'history': [{'role': m['role'], 'content': m['content']} for m in non_system[:-1]],
        'exaEnabled': False,
        'model': 'google/gemini-2.5-flash',
        'searchType': 'instant',
    }).encode()
    req = urllib.request.Request(EXA_URL, data=payload,
        headers={'Content-Type': 'application/json', 'Accept': 'text/event-stream'})
    full = ''; evt = None
    with urllib.request.urlopen(req, timeout=180) as resp:
        buf = b''
        for raw in resp:
            buf += raw
            lines = buf.split(b'\n'); buf = lines.pop()
            for line in lines:
                t = line.decode('utf-8', errors='replace').strip()
                if not t: continue
                if t.startswith('event:'): evt = t[6:].strip()
                elif t.startswith('data:') and evt == 'content':
                    try: full += json.loads(t[5:].strip()).get('content', '')
                    except: pass
    return _strip_followups(full)

class _TS(ThreadingMixIn, HTTPServer): daemon_threads = True
class _Handler(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def do_GET(self):
        if self.path == '/v1/models':
            b = json.dumps({'object': 'list', 'data': [{'id': 'manimator', 'object': 'model', 'created': 0, 'owned_by': 'exa'}]}).encode()
            self.send_response(200); self.send_header('Content-Type', 'application/json'); self.end_headers(); self.wfile.write(b)
        else: self.send_response(404); self.end_headers()
    def do_POST(self):
        if self.path != '/v1/chat/completions': self.send_response(404); self.end_headers(); return
        n = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(n))
        try: content = _collect_exa_stream(body.get('messages', []))
        except Exception as e:
            self.send_response(500); self.send_header('Content-Type', 'application/json'); self.end_headers()
            self.wfile.write(json.dumps({'error': str(e)}).encode()); return
        rb = json.dumps({'id': 'chatcmpl-local', 'object': 'chat.completion', 'model': 'manimator',
            'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': content}, 'finish_reason': 'stop'}],
            'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}}).encode()
        self.send_response(200); self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(rb))); self.end_headers(); self.wfile.write(rb)

try:
    srv = _TS(('127.0.0.1', LOCAL_PORT), _Handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    print(f'LLM proxy on :{LOCAL_PORT}')
except OSError:
    print(f'Port {LOCAL_PORT} already bound — server already running')

In [ ]:
import subprocess, pathlib, shutil, os, json, urllib.request, re
import IPython.display

def call_llm(messages):
    payload = json.dumps({'model': 'manimator', 'messages': messages, 'max_tokens': 2048}).encode()
    req = urllib.request.Request(f'http://127.0.0.1:{LOCAL_PORT}/v1/chat/completions',
        data=payload, headers={'Content-Type': 'application/json', 'Authorization': 'Bearer dummy'})
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content'].strip()

def plan_animation(user_prompt):
    raw = call_llm([
        {'role': 'system', 'content': (
            'You are a Manim animation planner. Given a topic, output ONLY a numbered scene plan — '
            'no questions, no clarifications, no preamble, no closing remarks, no code blocks. '
            'Scale the number of scenes to match the requested length: '
            'short/quick → 3-4 scenes, default → 5-6 scenes, long/detailed/comprehensive → 8-12 scenes. '
            'For each scene describe in plain text: what shapes/colors appear, how they move, what equations or text labels show. '
            'No code, no markdown fences, no backticks — just a numbered list of plain English descriptions. '
            'Never ask the user anything. Never say "I need more info". Just write the plan.'
        )},
        {'role': 'user', 'content': user_prompt},
    ])
    raw = re.sub(r'```.*?```', '', raw, flags=re.DOTALL).strip()
    for marker in ['Would you like', 'Do you want', 'Should I', 'Could you clarify',
                   'Can you provide', 'What specific', 'How long', 'Any specific',
                   'Please let me know', 'Let me know']:
        for sep in ('\n\n', '\n'):
            idx = raw.find(sep + marker)
            if idx >= 0:
                raw = raw[:idx]
    return raw.strip()

def run_aider(task, cwd, env, extra_files=None):
    files = ['plan.md'] + (extra_files or [])
    return subprocess.run(
        ['aider',
         '--model', 'openai/manimator',
         '--openai-api-base', f'http://127.0.0.1:{LOCAL_PORT}/v1',
         '--openai-api-key', 'dummy',
         '--yes-always', '--no-stream', '--no-auto-commits', '--no-git',
         '--no-show-model-warnings', '--no-check-update',
         '--message', task,
         *files],
        cwd=str(cwd), text=True, timeout=600, env=env,
    )

def try_render(cwd):
    r = subprocess.run(
        'python3 -m manim -pql --disable_caching scene.py AnimScene 2>&1',
        shell=True, cwd=str(cwd), capture_output=True, text=True
    )
    output = (r.stdout + r.stderr).strip()
    return r.returncode == 0, output

def run_manimator(user_prompt):
    print(f'Planning: {user_prompt!r}')
    plan = plan_animation(user_prompt)
    print('\n=== Scene Plan ===\n' + plan + '\n' + '=' * 40)

    for f in MANIM_OUTPUT.glob('*.py'):
        f.unlink()
    for f in MANIM_OUTPUT.glob('*.md'):
        f.unlink()

    (MANIM_OUTPUT / 'plan.md').write_text(
        f"# Animation Plan\n\n## Original request\n{user_prompt}\n\n## Scenes\n{plan}\n"
    )

    aider_env = {
        **os.environ,
        'OPENAI_API_KEY': 'dummy',
        'GIT_AUTHOR_NAME': 'aider', 'GIT_AUTHOR_EMAIL': 'aider@colab',
        'GIT_COMMITTER_NAME': 'aider', 'GIT_COMMITTER_EMAIL': 'aider@colab',
    }

    # Initial generation
    print('\n=== Aider: generating code ===')
    run_aider(
        "Read plan.md and implement the animation as a Manim project.\n\n"
        "Write files in this order:\n"
        "1. All helper modules first (objects.py, helpers.py, etc.) with all reusable classes/functions\n"
        "2. scene.py last — imports from helpers, defines AnimScene(Scene) with construct()\n\n"
        "Do not leave any file empty. scene.py runs with:\n"
        "  python3 -m manim -pql --disable_caching scene.py AnimScene\n\n"
        "Rules:\n"
        "- AnimScene(Scene) in scene.py\n"
        "- MathTex(r'...') for all equations and math symbols\n"
        "- Text() only for plain prose labels\n"
        "- Arrow(start=..., end=...) — never left=/right= kwargs\n"
        "- Make it visually complete and polished",
        MANIM_OUTPUT, aider_env
    )

    # Fix loop — we always test ourselves and feed errors back
    MAX_FIXES = 5
    for attempt in range(MAX_FIXES):
        py_files = [f.name for f in MANIM_OUTPUT.glob('*.py')]
        ok, output = try_render(MANIM_OUTPUT)
        print(f'\n--- Render attempt {attempt+1}: {"OK" if ok else "FAILED"} ---')
        if not ok:
            print(output[-1000:])
        if ok:
            break
        print(f'Sending error to aider for fix...')
        run_aider(
            f"Running `python3 -m manim -pql --disable_caching scene.py AnimScene` failed with:\n\n{output[-2000:]}\n\nFix all errors.",
            MANIM_OUTPUT, aider_env, extra_files=py_files
        )
    else:
        print('Max fix attempts reached.')

    videos = [v for v in MANIM_OUTPUT.rglob('*.mp4') if v.stat().st_size > 50_000]
    if not videos:
        print('No valid video rendered.')
        return

    dest = BASE / 'animation.mp4'
    shutil.copy(sorted(videos, key=lambda p: p.stat().st_mtime)[-1], dest)
    shutil.rmtree(MANIM_OUTPUT / 'media', ignore_errors=True)

    print(f'Video: {dest} ({dest.stat().st_size:,} bytes)')
    IPython.display.display(IPython.display.Video(str(dest), embed=True, html_attributes='controls width="720"'))

print('run_manimator() ready')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

prompt_box = widgets.Textarea(
    placeholder="e.g. Explain how a Fourier series builds up a square wave",
    layout=widgets.Layout(width="700px", height="100px"),
)
btn = widgets.Button(description="Generate", button_style="primary")
status = widgets.Output()

def on_generate(b):
    with status:
        status.clear_output()
        if not prompt_box.value.strip():
            print("Please enter a prompt.")
            return
        run_manimator(prompt_box.value.strip())

btn.on_click(on_generate)
display(prompt_box, btn, status)